# Домашнее задание №2

## Аффинные и проективные преобразования, RANSAC

### Цель

Научиться описывать геометрические преобразования изображения матрицами в однородных координатах, оценивать преобразование по точечным соответствиям и понимать, почему метод наименьших квадратов теряет работоспособность при наличии выбросов, а RANSAC — нет. Результат работы — не единичная удачная гомография, а измеренная зависимость точности оценки от доли выбросов для двух методов.

[Методические указания блока](README.md) · [Общие МУ](../../../docs/guidelines-students.md) · [Рубрика оценивания](../teachers-assessment/README.md)

## 1. Что используется в работе

| Библиотека | Роль в работе |
|---|---|
| `opencv-python` (`cv2`) | `warpAffine`, `warpPerspective`, `getPerspectiveTransform`, `findHomography` |
| `numpy` | матрицы преобразований, SVD, генерация соответствий |
| `scikit-image` | тестовые изображения |
| `matplotlib` | визуализация преобразований и графиков серий |
| `pandas` | журнал экспериментов и сводные таблицы |

Данные: `skimage.data` (`checkerboard`, `text`, `astronaut`) и синтетическое изображение-мишень с известными координатами узлов, генерируемое кодом. Соответствия точек для оценки гомографии синтезируются с **контролируемой** долей выбросов — это единственный способ построить корректную серию: истинная матрица известна, поэтому ошибка оценки измерима.

Интернет не требуется. Заполните шапку работы (ФИО, группа, версии, seed) согласно п. 1 [общих МУ](../../../docs/guidelines-students.md).

In [ ]:
# Служебная ячейка: импорты, версии, seed.
import time

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import skimage
from skimage import data as skdata

SEED = 42
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

AUTHOR = {"fio": "", "group": "", "work": "ДЗ2"}   # TODO: заполните

VERSIONS = {
    "opencv": cv2.__version__,
    "numpy": np.__version__,
    "scikit-image": skimage.__version__,
    "pandas": pd.__version__,
    "seed": SEED,
}
VERSIONS

## 2. Краткая теоретическая справка

### 2.1. Однородные координаты

Точка $(x, y)$ представляется вектором $\tilde{p} = (x, y, 1)^{\mathsf{T}}$, определённым с точностью до ненулевого множителя: $\tilde{p} \sim \lambda \tilde{p}$. Переход обратно — деление на третью координату: $(x, y) = (\tilde{x}/\tilde{w},\ \tilde{y}/\tilde{w})$. Однородные координаты позволяют записать сдвиг (не линейный в $\mathbb{R}^2$) как матричное умножение и представить композицию преобразований произведением матриц.

### 2.2. Аффинные преобразования

$$
T = \begin{pmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \\ 0 & 0 & 1 \end{pmatrix},\quad
R = \begin{pmatrix} \cos\theta & -\sin\theta & 0 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{pmatrix},\quad
S = \begin{pmatrix} s_x & 0 & 0 \\ 0 & s_y & 0 \\ 0 & 0 & 1 \end{pmatrix},\quad
H_{sh} = \begin{pmatrix} 1 & m & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{pmatrix}
$$

Общий вид аффинного преобразования:

$$ A = \begin{pmatrix} a_{11} & a_{12} & a_{13} \\ a_{21} & a_{22} & a_{23} \\ 0 & 0 & 1 \end{pmatrix} $$

— **6 степеней свободы**, для оценки достаточно **3 пар** точек в общем положении. Аффинное преобразование сохраняет параллельность прямых и отношение длин отрезков на одной прямой.

Композиция некоммутативна: поворот вокруг центра изображения записывается как $T_{c} R T_{-c}$, и порядок сомножителей менять нельзя.

### 2.3. Проективное преобразование (гомография)

$$ \tilde{p}' \sim H \tilde{p}, \qquad H = \begin{pmatrix} h_{11} & h_{12} & h_{13} \\ h_{21} & h_{22} & h_{23} \\ h_{31} & h_{32} & h_{33} \end{pmatrix} $$

Матрица определена с точностью до масштаба, поэтому у гомографии **8 степеней свободы** и требуется **4 пары** точек, никакие три из которых не лежат на одной прямой. Гомография сохраняет прямые, но не параллельность; в явном виде

$$ x' = \frac{h_{11}x + h_{12}y + h_{13}}{h_{31}x + h_{32}y + h_{33}}, \qquad
   y' = \frac{h_{21}x + h_{22}y + h_{23}}{h_{31}x + h_{32}y + h_{33}} $$

### 2.4. Оценка гомографии: DLT и МНК

Каждая пара соответствий даёт два линейных уравнения относительно вектора $h \in \mathbb{R}^9$ (составленного из элементов $H$):

$$ A h = 0 $$

где $A$ — матрица размера $2n \times 9$. Решение — правый сингулярный вектор $A$, отвечающий наименьшему сингулярному числу (SVD). При $n > 4$ это оценка в смысле наименьших квадратов алгебраической ошибки. МНК минимизирует **сумму квадратов** невязок, поэтому один выброс с большой ошибкой может доминировать над всеми остальными наблюдениями: у квадратичной функции потерь нет точки насыщения.

### 2.5. RANSAC

Итеративная схема: (1) случайно выбрать минимальную выборку (4 пары для гомографии); (2) построить по ней модель; (3) посчитать число согласных наблюдений (инлаеров) — тех, у кого ошибка репроекции меньше порога $\tau$; (4) повторить и оставить модель с максимальной поддержкой; (5) переоценить модель по всем инлаерам.

Необходимое число итераций для вероятности успеха $p$ при доле выбросов $\varepsilon$ и размере минимальной выборки $s$:

$$ N = \frac{\log(1 - p)}{\log\bigl(1 - (1-\varepsilon)^s\bigr)} $$

Например, при $s = 4$, $p = 0.99$, $\varepsilon = 0.5$ требуется $N \approx 72$ итерации. Устойчивость RANSAC достигается тем, что модель строится по подмножеству, не содержащему выбросов, а выбросы влияют только на счётчик поддержки — их вклад ограничен единицей, а не квадратом ошибки.

### 2.6. Метрика точности оценки

Средняя ошибка репроекции по инлаерам:

$$ e = \frac{1}{|\mathcal{I}|}\sum_{i \in \mathcal{I}} \bigl\| \pi(\hat H \tilde{p}_i) - p'_i \bigr\|_2 $$

Дополнительно используется ошибка на углах изображения: $\frac{1}{4}\sum_{k} \|\pi(\hat H \tilde{c}_k) - \pi(H_{\text{true}} \tilde{c}_k)\|_2$ — она измеряет геометрическое расхождение оценки с истиной независимо от конкретного набора точек.

## 3. Задачи

Формулировка по [методическим указаниям блока](README.md).

1. Реализуйте и примените к изображению базовые аффинные преобразования (сдвиг, поворот, масштаб, скос) через матрицы преобразований.
2. Выполните проективное преобразование по четырём парам точек соответствия.
3. Оцените гомографию по зашумлённым соответствиям с помощью RANSAC и сравните с оценкой по методу наименьших квадратов.

**Ожидаемый результат:** визуализации преобразований, сравнение устойчивости МНК и RANSAC при разной доле выбросов.

**Что будет проверяться** ([рубрика](../teachers-assessment/README.md)): матрицы преобразований выписаны и применены явно (не только вызовы высокоуровневых функций); гомография по 4 точкам корректна; сравнение МНК и RANSAC проведено при **контролируемой доле выбросов** — это серия, а не один запуск. Сравнение методов на данных без выбросов не подтверждает ничего: там они совпадают.

## 4. Данные

Используются `checkerboard` (регулярная структура, на ней хорошо видны искажения), `text` и синтетическая мишень с точно известными координатами узлов сетки. Мишень нужна для проверки реализации: применив к её узлам матрицу аналитически и сравнив с результатом `warp`, вы отделите ошибку реализации от ошибки интерполяции.

In [ ]:
# Служебная ячейка: данные и визуализация. Изменять не требуется.

def make_grid_target(height: int = 400, width: int = 400, step: int = 50) -> tuple:
    '''Синтетическая мишень: сетка линий и узлы с известными координатами.

    Выход: (image uint8 [H, W], nodes float32 [N, 2] в порядке (x, y)).
    '''
    img = np.full((height, width), 255, dtype=np.uint8)
    for x in range(0, width, step):
        cv2.line(img, (x, 0), (x, height - 1), 60, 1)
    for y in range(0, height, step):
        cv2.line(img, (0, y), (width - 1, y), 60, 1)
    nodes = []
    for y in range(step, height - step + 1, step):
        for x in range(step, width - step + 1, step):
            cv2.circle(img, (x, y), 4, 0, -1)
            nodes.append((x, y))
    cv2.rectangle(img, (10, 10), (width - 11, height - 11), 0, 2)
    return img, np.array(nodes, dtype=np.float32)


TARGET, TARGET_NODES = make_grid_target()
IMAGES = {
    "checkerboard": (skdata.checkerboard()).astype(np.uint8),
    "text": (skdata.text()).astype(np.uint8),
    "grid_target": TARGET,
}


def show_row(images, titles=None, figsize_scale: float = 3.4) -> None:
    n = len(images)
    titles = titles or [""] * n
    fig, axes = plt.subplots(1, n, figsize=(figsize_scale * n, figsize_scale))
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap="gray", vmin=0, vmax=255) if img.ndim == 2 else ax.imshow(img)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def draw_points(image: np.ndarray, points: np.ndarray, color=(255, 0, 0), radius: int = 5):
    '''Нанести точки (x, y) на копию изображения. Возвращает RGB-копию.'''
    canvas = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB) if image.ndim == 2 else image.copy()
    for x, y in np.asarray(points, dtype=np.float64).reshape(-1, 2):
        cv2.circle(canvas, (int(round(x)), int(round(y))), radius, color, -1)
    return canvas


show_row([IMAGES["checkerboard"], IMAGES["text"], IMAGES["grid_target"]],
         ["checkerboard", "text", f"grid_target, узлов: {len(TARGET_NODES)}"])

In [ ]:
# Служебная ячейка: журнал экспериментов. Изменять не требуется.
RUNS: list = []


def log_run(**fields) -> dict:
    row = {"seed": SEED, **fields}
    if isinstance(row.get("params"), dict):
        row["params"] = ", ".join(f"{k}={v}" for k, v in row["params"].items())
    RUNS.append(row)
    return row


def runs_table(stage: str = None) -> pd.DataFrame:
    df = pd.DataFrame(RUNS)
    if stage is not None and not df.empty:
        df = df[df["stage"] == stage]
    return df.reset_index(drop=True)


def apply_h(H: np.ndarray, points: np.ndarray) -> np.ndarray:
    '''Применить матрицу 3x3 к точкам [N, 2] в однородных координатах.

    Выход: [N, 2] float64. Деление на третью координату выполняется здесь,
    поэтому функция работает и для аффинных, и для проективных матриц.
    '''
    pts = np.asarray(points, dtype=np.float64).reshape(-1, 2)
    homo = np.hstack([pts, np.ones((len(pts), 1))])
    projected = homo @ np.asarray(H, dtype=np.float64).T
    return projected[:, :2] / projected[:, 2:3]


print("Журнал инициализирован, записей:", len(RUNS))

## 5. Задание 1. Аффинные преобразования через матрицы

Требование рубрики: матрицы должны быть **выписаны явно**. Вызов `cv2.getRotationMatrix2D` без собственной записи матрицы поворота задание не закрывает — им можно только сверить свой результат.

Рабочий пример ниже показывает полный цикл на сдвиге: построение матрицы $3\times3$, преобразование координат узлов аналитически, применение к изображению через `cv2.warpAffine` (принимает верхние 2 строки матрицы), контроль совпадения аналитических и фактических координат.

In [ ]:
# Рабочий пример (рельсы): сдвиг, выписанный явно.
tx, ty = 60.0, -30.0
T = np.array([[1.0, 0.0, tx],
              [0.0, 1.0, ty],
              [0.0, 0.0, 1.0]])

warped = cv2.warpAffine(TARGET, T[:2, :], (TARGET.shape[1], TARGET.shape[0]),
                        flags=cv2.INTER_LINEAR, borderValue=255)
nodes_expected = apply_h(T, TARGET_NODES)

show_row([draw_points(TARGET, TARGET_NODES[:6]),
          draw_points(warped, nodes_expected[:6], color=(0, 128, 255))],
         ["исходная мишень", f"сдвиг tx={tx}, ty={ty}"])

print("Первые три узла до:", TARGET_NODES[:3].tolist())
print("Первые три узла после (аналитически):", np.round(nodes_expected[:3], 2).tolist())

In [ ]:
# TODO (задание 1.1): реализуйте построение матриц базовых преобразований.

def affine_matrix(kind: str, **params) -> np.ndarray:
    '''Построить матрицу аффинного преобразования 3x3 в однородных координатах.

    Вход:
        kind : "translate" | "rotate" | "scale" | "shear"
        params:
            translate : tx, ty
            rotate    : angle_deg, center=(cx, cy) — по умолчанию (0, 0)
            scale     : sx, sy
            shear     : mx, my
    Выход:
        np.ndarray float64 [3, 3], последняя строка (0, 0, 1).
    Требования:
        - поворот вокруг центра реализуется композицией T(c) R T(-c),
          а не подстановкой центра внутрь матрицы поворота;
        - функция не должна вызывать cv2.getRotationMatrix2D — он используется
          только для сверки результата.
    '''
    raise NotImplementedError


def compose(*matrices) -> np.ndarray:
    '''Композиция преобразований. Определите и задокументируйте порядок применения:
    compose(A, B) означает «сначала B, затем A» или наоборот — выберите и придерживайтесь.
    '''
    raise NotImplementedError

In [ ]:
# TODO (задание 1.2): примените преобразования и проверьте реализацию.
#
# Требования:
# 1. Показать сдвиг, поворот (вокруг центра изображения), масштаб и скос —
#    каждое отдельной визуализацией с подписью параметров.
# 2. Показать композицию не менее трёх преобразований и продемонстрировать
#    некоммутативность: compose(R, S) != compose(S, R) — численно и визуально.
# 3. Проверить корректность: сравнить свою матрицу поворота вокруг центра с
#    cv2.getRotationMatrix2D(center, angle, 1.0); разница по норме должна быть
#    порядка машинной точности. Расхождение означает ошибку в знаке угла,
#    порядке композиции или соглашении о направлении оси y.
# 4. Сравнить методы интерполяции (cv2.INTER_NEAREST и cv2.INTER_LINEAR)
#    на масштабировании и описать различие.
#
# Проверка через узлы мишени:
#   error = np.abs(apply_h(A, TARGET_NODES) - <координаты узлов на warp-изображении>)
# Для аналитической проверки достаточно сравнить apply_h(A, nodes) с ожидаемыми
# координатами, вычисленными вручную для одного-двух узлов.

# TODO: код заданий 1.2 и запись в журнал через log_run(stage="affine", ...)

## 6. Задание 2. Проективное преобразование по четырём парам точек

Четыре пары точек задают гомографию однозначно (8 уравнений на 8 степеней свободы). Условие применимости: никакие три из четырёх точек не должны лежать на одной прямой — иначе система вырождена, и `cv2.getPerspectiveTransform` вернёт матрицу с огромными значениями или предупреждение.

Рабочий пример: «съёмка под углом» — прямоугольник изображения отображается в произвольный выпуклый четырёхугольник. Обратное преобразование (выравнивание) — это уже прообраз ЛР2 блока.

In [ ]:
# Рабочий пример (рельсы): гомография по четырём углам.
h, w = TARGET.shape
src_quad = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])
dst_quad = np.float32([[40, 60], [w - 25, 15], [w - 60, h - 30], [15, h - 70]])

H_cv = cv2.getPerspectiveTransform(src_quad, dst_quad)
projected = cv2.warpPerspective(TARGET, H_cv, (w, h), borderValue=255)

print("H (cv2.getPerspectiveTransform), нормировано h33=1:")
print(np.round(H_cv / H_cv[2, 2], 5))
print("Проверка: углы src -> dst")
print(np.round(apply_h(H_cv, src_quad), 2))

show_row([draw_points(TARGET, src_quad), draw_points(projected, dst_quad, color=(0, 128, 255))],
         ["исходное, 4 опорные точки", "проективное преобразование"])

In [ ]:
# TODO (задание 2.1): реализуйте оценку гомографии по четырём парам точек (DLT).

def homography_from_points(src: np.ndarray, dst: np.ndarray) -> np.ndarray:
    '''Оценить гомографию по парам соответствий методом DLT.

    Вход:
        src : np.ndarray [N, 2], N >= 4 — точки исходного изображения (x, y)
        dst : np.ndarray [N, 2] — соответствующие точки целевого изображения
    Выход:
        np.ndarray float64 [3, 3], нормированная так, что H[2, 2] == 1
        (если H[2, 2] != 0).
    Порядок действий:
        1. Составить матрицу A размера 2N x 9: каждая пара даёт две строки.
        2. Найти правый сингулярный вектор A с наименьшим сингулярным числом
           (np.linalg.svd, последняя строка Vt).
        3. Собрать матрицу 3x3 и нормировать.
    Замечание: для устойчивости рекомендуется нормализация координат
    (перенос центра масс в начало и масштабирование до среднего расстояния
    sqrt(2)) с последующим обратным преобразованием матрицы. Реализуйте её и
    покажите разницу в точности — это отдельный пункт анализа.
    Запрещено вызывать cv2.getPerspectiveTransform или cv2.findHomography внутри.
    '''
    raise NotImplementedError


# TODO: сверьте свою реализацию с cv2.getPerspectiveTransform на src_quad/dst_quad.
# Ожидание: обе матрицы после нормировки h33=1 совпадают с точностью ~1e-8.
# TODO: примените своё преобразование к изображению (cv2.warpPerspective)
# и выполните обратное выравнивание через np.linalg.inv(H).
# TODO: покажите, что происходит при трёх парах точек и при четырёх точках,
# три из которых коллинеарны (этот вопрос задаётся на защите).

## 7. Задание 3. Данные для сравнения методов оценки

Для честного сравнения МНК и RANSAC истинная гомография должна быть известна. Служебная функция ниже генерирует набор соответствий по заданной матрице $H_{\text{true}}$ с двумя независимо управляемыми факторами:

- `noise_sigma` — гауссов шум локализации точек (присутствует у всех соответствий, это не выброс);
- `outlier_ratio` — доля **выбросов**, то есть пар, поставленных в соответствие ошибочно (координаты второй точки случайны и не связаны с первой).

Выброс — не «сильно зашумлённая точка», а точка из другого распределения. Смешивать эти два фактора в одной серии нельзя: меняйте по одному.

In [ ]:
# Служебная ячейка: генерация соответствий с контролируемой долей выбросов.

def random_homography(width: int, height: int, strength: float = 0.12,
                      seed: int = SEED) -> np.ndarray:
    '''Случайная, но разумная гомография: углы кадра смещаются не более чем на
    strength * размер кадра. Выход: [3, 3], H[2, 2] == 1.
    '''
    rng = np.random.default_rng(seed)
    src = np.float32([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]])
    shift = rng.uniform(-strength, strength, size=(4, 2)) * np.array([width, height])
    dst = (src + shift).astype(np.float32)
    H = cv2.getPerspectiveTransform(src, dst)
    return H / H[2, 2]


def make_correspondences(H_true: np.ndarray, n_points: int = 100,
                         noise_sigma: float = 1.0, outlier_ratio: float = 0.0,
                         width: int = 400, height: int = 400, seed: int = SEED) -> tuple:
    '''Сгенерировать пары соответствий по известной гомографии.

    Вход:
        H_true        : истинная матрица [3, 3]
        n_points      : общее число пар
        noise_sigma   : стандартное отклонение шума локализации, пиксели
        outlier_ratio : доля пар-выбросов в [0, 1)
    Выход:
        (src [N, 2] float64, dst [N, 2] float64, is_inlier [N] bool)
    '''
    rng = np.random.default_rng(seed)
    src = rng.uniform([0, 0], [width, height], size=(n_points, 2))
    dst = apply_h(H_true, src) + rng.normal(0.0, noise_sigma, size=(n_points, 2))

    n_outliers = int(round(outlier_ratio * n_points))
    is_inlier = np.ones(n_points, dtype=bool)
    if n_outliers > 0:
        idx = rng.choice(n_points, size=n_outliers, replace=False)
        dst[idx] = rng.uniform([0, 0], [width, height], size=(n_outliers, 2))
        is_inlier[idx] = False
    return src, dst, is_inlier


def reprojection_errors(H: np.ndarray, src: np.ndarray, dst: np.ndarray) -> np.ndarray:
    '''Поточечная ошибка репроекции ||pi(H p) - p'||, [N] float64.'''
    return np.linalg.norm(apply_h(H, src) - np.asarray(dst, dtype=np.float64), axis=1)


def corner_error(H_est: np.ndarray, H_true: np.ndarray,
                 width: int = 400, height: int = 400) -> float:
    '''Средняя ошибка на четырёх углах кадра, пиксели. Главная метрика серии:
    не зависит от конкретного набора точек и от доли выбросов в нём.
    '''
    corners = np.float32([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]])
    return float(np.mean(np.linalg.norm(apply_h(H_est, corners) - apply_h(H_true, corners), axis=1)))


H_TRUE = random_homography(400, 400, seed=SEED)
src_pts, dst_pts, inlier_flags = make_correspondences(H_TRUE, n_points=100,
                                                      noise_sigma=1.0, outlier_ratio=0.3)
print("Истинная гомография:\n", np.round(H_TRUE, 5))
print("Пар:", len(src_pts), "| выбросов:", int((~inlier_flags).sum()))

In [ ]:
# Рабочий пример (рельсы): одна конфигурация, эталонная реализация RANSAC из OpenCV.
# Это ориентир для проверки собственной реализации, а не замена ей.
H_ref, mask = cv2.findHomography(src_pts.reshape(-1, 1, 2), dst_pts.reshape(-1, 1, 2),
                                 method=cv2.RANSAC, ransacReprojThreshold=3.0,
                                 maxIters=2000, confidence=0.995)
mask = mask.ravel().astype(bool)

print("Ошибка на углах, cv2.RANSAC:", round(corner_error(H_ref / H_ref[2, 2], H_TRUE), 3), "px")
print("Найдено инлаеров:", int(mask.sum()), "из", len(src_pts),
      "| истинных инлаеров:", int(inlier_flags.sum()))
print("Верно распознано инлаеров:", int((mask & inlier_flags).sum()),
      "| выбросов, ошибочно принятых за инлаеры:", int((mask & ~inlier_flags).sum()))

plt.figure(figsize=(5, 5))
plt.scatter(dst_pts[inlier_flags, 0], dst_pts[inlier_flags, 1], s=12, label="инлаеры (истина)")
plt.scatter(dst_pts[~inlier_flags, 0], dst_pts[~inlier_flags, 1], s=18, marker="x",
            label="выбросы (истина)")
plt.gca().invert_yaxis()
plt.legend(fontsize=8)
plt.title("Соответствия в целевом кадре, доля выбросов 0.3", fontsize=10)
plt.tight_layout()
plt.show()

## 8. Задание 3. МНК против RANSAC

Реализуйте оба метода самостоятельно. `cv2.findHomography` используется только как эталон для сверки: если ваша RANSAC-оценка систематически хуже, ищите ошибку в пороге, числе итераций или в переоценке модели по инлаерам.

Порог $\tau$ и число итераций — параметры, а не константы. Число итераций считайте по формуле из раздела 2.5 либо задавайте фиксированным и обосновывайте.

**Ключевое требование к серии.** Один запуск при одной доле выбросов ничего не доказывает. Нужна зависимость ошибки от `outlier_ratio` (не менее 5 значений, включая 0), при нескольких повторах с разными seed — иначе результат RANSAC, метода стохастического, не отличим от случайности.

In [ ]:
# TODO (задание 3.1): оценка гомографии методом наименьших квадратов.

def estimate_homography_ls(src: np.ndarray, dst: np.ndarray) -> np.ndarray:
    '''Оценить гомографию по всем парам соответствий (DLT + SVD, без отбраковки).

    Вход:  src, dst — [N, 2], N >= 4
    Выход: [3, 3] float64, нормированная h33 = 1.
    Может использовать homography_from_points из задания 2.1.
    '''
    raise NotImplementedError

In [ ]:
# TODO (задание 3.2): собственная реализация RANSAC.

def ransac_homography(src: np.ndarray, dst: np.ndarray, threshold: float = 3.0,
                      max_iters: int = 2000, confidence: float = 0.99,
                      seed: int = SEED) -> tuple:
    '''Робастная оценка гомографии.

    Вход:
        src, dst   : [N, 2] соответствия
        threshold  : порог ошибки репроекции для инлаера, пиксели
        max_iters  : верхняя граница числа итераций
        confidence : требуемая вероятность успеха p (для адаптивной остановки)
    Выход:
        (H [3, 3] float64 с h33 = 1, inlier_mask [N] bool, n_iters_used int)
    Порядок действий:
        1. Итерация: выбрать 4 случайные пары, оценить модель.
        2. Посчитать ошибки репроекции для всех пар, отметить инлаеры (err < threshold).
        3. Запомнить модель с наибольшим числом инлаеров.
        4. Опционально: адаптивно пересчитать требуемое число итераций
           N = log(1 - p) / log(1 - (1 - eps)^4) по текущей оценке доли выбросов eps.
        5. Переоценить модель по всем инлаерам лучшей гипотезы (это обязательный шаг:
           модель по 4 точкам использует шум только четырёх наблюдений).
    Граничные случаи: вырожденная минимальная выборка (коллинеарные точки),
    отсутствие модели с 4+ инлаерами — обработайте явно.
    '''
    raise NotImplementedError

In [ ]:
# TODO (задание 3.3): контролируемая серия по доле выбросов.
#
# Обязательный план серии:
#   outlier_ratios = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
#   repeats        = 5 разных seed на каждую точку (RANSAC стохастичен)
#   методы         = {"ls", "ransac_own", "cv_ransac"}
#   фиксировано    = n_points, noise_sigma, H_TRUE, threshold
# Метрики в журнал: corner_error (главная), медиана ошибки репроекции по истинным
# инлаерам, доля верно найденных инлаеров, время выполнения.
#
# Отдельная серия (меняется другой фактор при outlier_ratio = const):
#   noise_sigma = [0.5, 1.0, 2.0, 4.0] — покажет, что при отсутствии выбросов
#   МНК не уступает RANSAC, а при их наличии картина меняется.
#
# Каркас:
# for ratio in outlier_ratios:
#     for rep in range(repeats):
#         s, d, flags = make_correspondences(H_TRUE, n_points=100, noise_sigma=1.0,
#                                            outlier_ratio=ratio, seed=SEED + rep)
#         for method in (...):
#             t0 = time.perf_counter()
#             H_est, inliers = ...
#             log_run(stage="homography", method=method, outlier_ratio=ratio, rep=rep,
#                     corner_err=corner_error(H_est, H_TRUE), ms=(time.perf_counter()-t0)*1e3,
#                     inlier_recall=..., params={"threshold": 3.0, "n_points": 100})

# TODO: код серии

runs_table("homography")

In [ ]:
# TODO (задание 3.4): график зависимости ошибки от доли выбросов.
#
# Требования к графику:
# - по оси x — доля выбросов, по оси y — ошибка на углах (пиксели);
# - логарифмическая шкала по y (ошибка МНК растёт на порядки);
# - для каждого метода — медиана по повторам и разброс (например, межквартильный);
# - подпись осей и легенда обязательны.
#
# Каркас построения из журнала:
# df = runs_table("homography")
# stats = df.groupby(["method", "outlier_ratio"])["corner_err"].median().unstack(0)
# stats.plot(marker="o", logy=True)

# TODO: код графика

## Отчёт

**Таблица 1.** Аффинные преобразования: преобразование → параметры → матрица (или её норма) → визуальный результат. Достаточно перечисления с иллюстрациями.

**Таблица 2.** Проверка гомографии по 4 точкам: собственная реализация против `cv2.getPerspectiveTransform`, норма разности, ошибка репроекции опорных точек.

**Таблица 3 — главная.** Доля выбросов × метод → медианная ошибка на углах, разброс, доля найденных инлаеров, время. Строится кодом ниже из журнала.

Разделяйте наблюдение, интерпретацию и вывод (п. 2 [общих МУ](../../../docs/guidelines-students.md)):

- **наблюдение** — «при доле выбросов 0.3 медианная ошибка на углах: МНК 84 px, RANSAC 1.2 px»;
- **интерпретация** — «квадратичная функция потерь МНК не ограничивает вклад одного выброса»;
- **вывод с границами** — «в проверенном диапазоне до 0.5 выбросов при σ = 1 px и 100 парах RANSAC с порогом 3 px даёт ошибку углов ниже 2 px; поведение при доле выбросов выше 0.5 не проверялось».

In [ ]:
# Сводные таблицы из журнала.
homography_runs = runs_table("homography")

if not homography_runs.empty:
    summary = homography_runs.pivot_table(
        index="outlier_ratio", columns="method",
        values="corner_err", aggfunc=["median", "std"]).round(3)
    display(summary)
    display(homography_runs.groupby("method")[["ms"]].mean().round(2))
else:
    print("Журнал пуст: выполните задание 3.3.")

affine_runs = runs_table("affine")
if not affine_runs.empty:
    display(affine_runs)

### Выводы

**Наблюдения**

1.
2.
3.

**Интерпретация**

1.
2.

**Выводы и границы применимости**

1.
2.

**Анализ отказов.** Разберите не менее двух случаев: (а) доля выбросов, при которой ваша реализация RANSAC начинает давать неверную модель; (б) влияние порога $\tau$ — что происходит при слишком малом и слишком большом значении; (в) при желании — вырожденные конфигурации точек.

**Использование сторонних материалов и LLM.** Укажите источники (п. 5 общих МУ).

## Контрольные вопросы

Из [списка вопросов блока](README.md#контрольные-вопросы-блока), относящиеся к этой работе:

5. Сколько степеней свободы у аффинного и проективного преобразований и сколько пар точек нужно для их оценки?
6. Как работает RANSAC и почему он устойчив к выбросам?

Дополнительно к защите: покажите в коде, где именно оценивается гомография; что произойдёт при трёх парах точек и что — при четырёх, три из которых коллинеарны?

## Чек-лист перед сдачей

Полный список — в [общих МУ, п. 6](../../../docs/guidelines-students.md#6-чек-лист-перед-сдачей). Специфика ДЗ2:

- [ ] Ноутбук исполняется сверху вниз без ошибок после `Restart & Run All`.
- [ ] Матрицы сдвига, поворота, масштаба и скоса выписаны явно в коде.
- [ ] Показана композиция преобразований и её некоммутативность.
- [ ] Гомография по 4 точкам реализована самостоятельно и сверена с OpenCV.
- [ ] МНК и RANSAC реализованы и сравнены на **одних и тех же** соответствиях.
- [ ] Сравнение проведено при контролируемой доле выбросов: не менее 5 значений, включая 0, с повторами.
- [ ] Есть график зависимости ошибки от доли выбросов.
- [ ] Указаны порог RANSAC, число итераций и способ их выбора.
- [ ] Наблюдения отделены от интерпретаций, границы выводов указаны.